# Motion Comparison: G1 Robot · SOMA Uniform · SOMA Proportional

Three representations of the **same motion** across three contrasting example sequences:

| Track | Type | Body model | Format |
|---|---|---|---|
| **G1 Robot** | Physical robot | Unitree G1 (29 DOF) | CSV — root pose + joint angles (°) |
| **SOMA Uniform** | MoCap retarget | Generic human template | BVH — Euler rotations |
| **SOMA Proportional** | MoCap retarget | Actor-specific proportions | BVH — Euler rotations |

**Method:**  
- SOMA → BVH forward kinematics (correct parent-stack parser, Rz·Ry·Rx accumulation)  
- G1 → robot FK (Z-up, 29-DOF chain, cm→mm, converted to Y-up for display)

**Examples:**
| Cell | Motion | Frames | Description | Output |
|---|---|---|---|---|
| Cell 5 | `jump_and_land_heavy_001__A001` | 1463 (366 sampled) | vertical jump & land | `g1_soma_comparison_fk.gif` |
| Cell 6 | `cartwheel_R_001__A135` | 851 (213 sampled) | full-body cartwheel | `g1_soma_cartwheel_fk.gif` |
| Cell 7 | `sit_on_heels_loop_009__A548` | 619 (155 sampled) | kneeling / sit on heels | `g1_soma_sit_on_heels_fk.gif` |

**Layout per GIF:** G1 Robot (green) · SOMA Uniform (blue) · SOMA Proportional (red) · Uniform vs Proportional overlay

In [64]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import pandas as pd
import glob, os, io, time, warnings
from matplotlib.lines import Line2D
from PIL import Image
warnings.filterwarnings('ignore')

# ── Dataset paths ─────────────────────────────────────────────────────────
DATA_ROOT             = '/home/grease/ego_dataset/work_bearlu/data/bones-studio-seed'
SOMA_UNIFORM_DIR      = os.path.join(DATA_ROOT, 'soma_uniform',      'bvh')
SOMA_PROPORTIONAL_DIR = os.path.join(DATA_ROOT, 'soma_proportional', 'bvh')
G1_DIR                = os.path.join(DATA_ROOT, 'g1',                'csv')
MOTION_NAME           = 'jump_and_land_heavy_001__A001'

# ── Locate matching files ─────────────────────────────────────────────────
uniform_files      = sorted(glob.glob(os.path.join(SOMA_UNIFORM_DIR,      '**/*.bvh'), recursive=True))
proportional_files = sorted(glob.glob(os.path.join(SOMA_PROPORTIONAL_DIR, '**/*.bvh'), recursive=True))
g1_files           = sorted(glob.glob(os.path.join(G1_DIR,               '**/*.csv'), recursive=True))

bvh_uniform      = next(f for f in uniform_files      if MOTION_NAME in f)
bvh_proportional = next(f for f in proportional_files if MOTION_NAME in f)
g1_csv           = next(f for f in g1_files           if MOTION_NAME in f)

print(f'Uniform      : {bvh_uniform}')
print(f'Proportional : {bvh_proportional}')
print(f'G1 Robot     : {g1_csv}')

Uniform      : /home/grease/ego_dataset/work_bearlu/data/bones-studio-seed/soma_uniform/bvh/210531/jump_and_land_heavy_001__A001.bvh
Proportional : /home/grease/ego_dataset/work_bearlu/data/bones-studio-seed/soma_proportional/bvh/210531/jump_and_land_heavy_001__A001.bvh
G1 Robot     : /home/grease/ego_dataset/work_bearlu/data/bones-studio-seed/g1/csv/210531/jump_and_land_heavy_001__A001.csv


In [62]:
# ── BVH parser ────────────────────────────────────────────────────────────
def parse_bvh(path):
    """
    Parse a BVH file and return the skeleton hierarchy plus all motion frames.

    Critical correctness rule: every JOINT node (including HeadEnd, ThumbEnd …)
    is pushed onto parent_stack.  Only 'End Site' leaf-marker blocks are consumed
    without touching the stack.  Skipping named joints corrupts the stack and
    gives wrong parents for ~40 % of the skeleton.

    Returns
    -------
    joints  : list of joint names (DFS order)
    offsets : {name: np.array([x,y,z])}  T-pose bone offset from parent
    channels: {name: {'start': int, 'types': [str,…]}}
    parents : {name: parent_name or None}
    frames  : np.ndarray  shape (F, total_channels)
    """
    with open(path) as f:
        lines = f.read().split('\n')

    joints, offsets, channels, parents = [], {}, {}, {}
    stack, ch_idx, i = [], 0, 0

    while i < len(lines):
        line = lines[i].strip()
        if line.startswith('ROOT ') or line.startswith('JOINT '):
            name = line.split()[1]
            joints.append(name)
            parents[name] = stack[-1] if stack else None
            stack.append(name)
        elif line.startswith('OFFSET') and stack:
            p = line.split()
            offsets[stack[-1]] = np.array([float(p[1]), float(p[2]), float(p[3])])
        elif line.startswith('CHANNELS') and stack:
            p = line.split(); n = int(p[1])
            channels[stack[-1]] = {'start': ch_idx, 'types': p[2:2+n]}
            ch_idx += n
        elif line.startswith('End Site'):       # leaf marker — consume without stack change
            i += 1
            while i < len(lines):
                if lines[i].strip() == '}':
                    break
                i += 1
        elif line == '}' and stack:
            stack.pop()
        elif line.strip() == 'MOTION':
            break
        i += 1

    mo = next(k for k, l in enumerate(lines) if l.strip() == 'MOTION')
    nf = int(lines[mo+1].split()[1])
    frame_data = []
    for k in range(mo+3, mo+3+nf):
        vals = lines[k].split()
        if vals:
            frame_data.append([float(v) for v in vals])
    return joints, offsets, channels, parents, np.array(frame_data)


# ── Forward kinematics ────────────────────────────────────────────────────
def _rot(angles, types):
    """Rotation matrix from BVH Euler channels (applied in listed order)."""
    def Rx(a): c,s=np.cos(a),np.sin(a); return np.array([[1,0,0],[0,c,-s],[0,s,c]])
    def Ry(a): c,s=np.cos(a),np.sin(a); return np.array([[c,0,s],[0,1,0],[-s,0,c]])
    def Rz(a): c,s=np.cos(a),np.sin(a); return np.array([[c,-s,0],[s,c,0],[0,0,1]])
    R = np.eye(3)
    for ang, t in zip(angles, types):
        a = np.radians(ang)
        if t == 'Xrotation': R = R @ Rx(a)
        elif t == 'Yrotation': R = R @ Ry(a)
        elif t == 'Zrotation': R = R @ Rz(a)
    return R

def fk(joints, offsets, channels, parents, frame_data):
    """
    Compute world-space 3-D position for every joint.
    world_pos[j] = world_pos[parent] + world_rot[parent] @ offset[j]
    world_rot[j] = world_rot[parent] @ Rz(z) @ Ry(y) @ Rx(x)
    """
    wp, wr = {}, {}
    for j in joints:
        if j not in channels:
            wp[j] = wp.get(parents.get(j), np.zeros(3)).copy()
            wr[j] = wr.get(parents.get(j), np.eye(3)).copy()
            continue
        ch = channels[j]; start, types = ch['start'], ch['types']
        pos_v = [None]*3; rot_a, rot_t = [], []
        for k, t in enumerate(types):
            v = frame_data[start+k]
            if 'position' in t.lower(): pos_v['XYZ'.index(t[0].upper())] = v
            else: rot_a.append(v); rot_t.append(t)
        local_rot = _rot(rot_a, rot_t)
        parent = parents[j]
        if parent is None:
            wp[j] = np.array([v if v else 0.0 for v in pos_v])
            wr[j] = local_rot
        else:
            local_pos = np.array([v if v else 0.0 for v in pos_v])
            wp[j] = wp.get(parent, np.zeros(3)) + wr.get(parent, np.eye(3)) @ offsets.get(j, np.zeros(3)) + local_pos
            wr[j] = wr.get(parent, np.eye(3)) @ local_rot
    return wp


# ── Body joints used for visualisation (no fingers / face) ────────────────
VIZ_JOINTS = [
    'Hips',
    'Spine1', 'Spine2', 'Chest', 'Neck1', 'Head',
    'LeftShoulder',  'LeftArm',  'LeftForeArm',  'LeftHand',
    'RightShoulder', 'RightArm', 'RightForeArm', 'RightHand',
    'LeftLeg',  'LeftShin',  'LeftFoot',
    'RightLeg', 'RightShin', 'RightFoot',
]

BONES = [
    # Spine
    ('Hips','Spine1'), ('Spine1','Spine2'), ('Spine2','Chest'),
    ('Chest','Neck1'), ('Neck1','Head'),
    # Arms
    ('Chest','LeftShoulder'),  ('LeftShoulder','LeftArm'),   ('LeftArm','LeftForeArm'),   ('LeftForeArm','LeftHand'),
    ('Chest','RightShoulder'), ('RightShoulder','RightArm'), ('RightArm','RightForeArm'), ('RightForeArm','RightHand'),
    # Legs
    ('Hips','LeftLeg'),  ('LeftLeg','LeftShin'),   ('LeftShin','LeftFoot'),
    ('Hips','RightLeg'), ('RightLeg','RightShin'), ('RightShin','RightFoot'),
]


# ── Load BVH and run FK for every sampled frame ───────────────────────────
print('Parsing BVH files…')
j_u, o_u, c_u, p_u, frames_u = parse_bvh(bvh_uniform)
j_p, o_p, c_p, p_p, frames_p = parse_bvh(bvh_proportional)
print(f'  Uniform      : {len(j_u)} joints, {len(frames_u)} frames')
print(f'  Proportional : {len(j_p)} joints, {len(frames_p)} frames')

FRAME_STEP = 4
sampled = list(range(0, min(len(frames_u), len(frames_p)), FRAME_STEP))

def fk_batch(viz, joints, offsets, channels, parents, all_frames, sampled):
    """Return (N_sampled, N_viz_joints, 3) array of world positions."""
    out = np.zeros((len(sampled), len(viz), 3))
    for fi, f in enumerate(sampled):
        wp = fk(joints, offsets, channels, parents, all_frames[f])
        for ji, name in enumerate(viz):
            out[fi, ji] = wp.get(name, np.zeros(3))
    return out

print(f'Running FK for {len(sampled)} sampled frames…')
pos_u = fk_batch(VIZ_JOINTS, j_u, o_u, c_u, p_u, frames_u, sampled)   # (F, J, 3)
pos_p = fk_batch(VIZ_JOINTS, j_p, o_p, c_p, p_p, frames_p, sampled)   # (F, J, 3)

# Floor-align: shift each skeleton so minimum foot Y = 0
foot_idx = [VIZ_JOINTS.index(n) for n in ('LeftFoot', 'RightFoot')]
pos_u[:, :, 1] -= pos_u[:, foot_idx, 1].min()
pos_p[:, :, 1] -= pos_p[:, foot_idx, 1].min()

print(f'  Uniform      Y-range : [{pos_u[:,:,1].min():.0f}, {pos_u[:,:,1].max():.0f}] mm')
print(f'  Proportional Y-range : [{pos_p[:,:,1].min():.0f}, {pos_p[:,:,1].max():.0f}] mm')
print('FK complete ✓')

Parsing BVH files…
  Uniform      : 78 joints, 1463 frames
  Proportional : 78 joints, 1463 frames
Running FK for 366 sampled frames…
  Uniform      Y-range : [0, 173] mm
  Proportional Y-range : [0, 149] mm
FK complete ✓
  Uniform      Y-range : [0, 173] mm
  Proportional Y-range : [0, 149] mm
FK complete ✓


In [80]:
# ── G1 Robot Forward Kinematics ───────────────────────────────────────────
#
# G1 CSV columns:
#   root_translateX/Y/Z  — root position in cm, Z-up convention
#   root_rotateX/Y/Z     — root orientation in degrees (ZYX applied)
#   *_joint_dof          — single-DOF joint angles in degrees
#
# A fixed yaw derotation (from the first frame) is applied to every frame
# so the skeleton always starts facing the viewer.  For the jump motion
# (yaw span ≈ 14°) this is equivalent to per-frame derotation.
# For the cartwheel (yaw span ≈ 133°), the body's actual rotation becomes
# visible in the display instead of being silently cancelled each frame.

# ── G1 skeleton: (name, parent, offset_mm_in_parent_Zup, [(axis, col)]) ──
G1_CHAIN = [
    # name               parent            offset(mm)       [(axis, dof_col)]
    ('pelvis',           None,             [ 0,   0,   0],  []),
    ('left_hip',         'pelvis',         [ 0, 115, -60],  [('Y','left_hip_pitch_joint_dof'),
                                                              ('X','left_hip_roll_joint_dof'),
                                                              ('Z','left_hip_yaw_joint_dof')]),
    ('left_knee',        'left_hip',       [ 0,   0,-330],  [('Y','left_knee_joint_dof')]),
    ('left_ankle',       'left_knee',      [ 0,   0,-350],  [('Y','left_ankle_pitch_joint_dof'),
                                                              ('X','left_ankle_roll_joint_dof')]),
    ('left_foot',        'left_ankle',     [80,   0, -80],  []),
    ('right_hip',        'pelvis',         [ 0,-115, -60],  [('Y','right_hip_pitch_joint_dof'),
                                                              ('X','right_hip_roll_joint_dof'),
                                                              ('Z','right_hip_yaw_joint_dof')]),
    ('right_knee',       'right_hip',      [ 0,   0,-330],  [('Y','right_knee_joint_dof')]),
    ('right_ankle',      'right_knee',     [ 0,   0,-350],  [('Y','right_ankle_pitch_joint_dof'),
                                                              ('X','right_ankle_roll_joint_dof')]),
    ('right_foot',       'right_ankle',    [80,   0, -80],  []),
    ('waist',            'pelvis',         [ 0,   0, 100],  [('Z','waist_yaw_joint_dof'),
                                                              ('X','waist_roll_joint_dof'),
                                                              ('Y','waist_pitch_joint_dof')]),
    ('chest',            'waist',          [ 0,   0, 130],  []),
    ('neck',             'chest',          [ 0,   0, 160],  []),
    ('left_shoulder',    'chest',          [ 0, 215,   0],  [('Y','left_shoulder_pitch_joint_dof'),
                                                              ('X','left_shoulder_roll_joint_dof'),
                                                              ('Z','left_shoulder_yaw_joint_dof')]),
    ('left_elbow',       'left_shoulder',  [ 0,   0,-260],  [('Y','left_elbow_joint_dof')]),
    ('left_wrist',       'left_elbow',     [ 0,   0,-250],  [('X','left_wrist_roll_joint_dof'),
                                                              ('Y','left_wrist_pitch_joint_dof'),
                                                              ('Z','left_wrist_yaw_joint_dof')]),
    ('right_shoulder',   'chest',          [ 0,-215,   0],  [('Y','right_shoulder_pitch_joint_dof'),
                                                              ('X','right_shoulder_roll_joint_dof'),
                                                              ('Z','right_shoulder_yaw_joint_dof')]),
    ('right_elbow',      'right_shoulder', [ 0,   0,-260],  [('Y','right_elbow_joint_dof')]),
    ('right_wrist',      'right_elbow',    [ 0,   0,-250],  [('X','right_wrist_roll_joint_dof'),
                                                              ('Y','right_wrist_pitch_joint_dof'),
                                                              ('Z','right_wrist_yaw_joint_dof')]),
]

G1_VIZ   = [name for name, *_ in G1_CHAIN]
G1_BONES = [
    ('pelvis','left_hip'),    ('left_hip','left_knee'),     ('left_knee','left_ankle'),   ('left_ankle','left_foot'),
    ('pelvis','right_hip'),   ('right_hip','right_knee'),   ('right_knee','right_ankle'), ('right_ankle','right_foot'),
    ('pelvis','waist'),       ('waist','chest'),             ('chest','neck'),
    ('chest','left_shoulder'),  ('left_shoulder','left_elbow'),  ('left_elbow','left_wrist'),
    ('chest','right_shoulder'), ('right_shoulder','right_elbow'),('right_elbow','right_wrist'),
]


# ── FK for one G1 frame ───────────────────────────────────────────────────
def g1_fk(row, fixed_yaw_rad=None):
    """
    Compute G1 world joint positions.

    fixed_yaw_rad : constant yaw (radians) to derotate by.
                    If None the per-frame yaw is used (fine when yaw barely changes).
                    Pass the first-frame yaw for motions where the body yaws a lot
                    (e.g. cartwheel) so the rotation is visible instead of cancelled.
    """
    def Rx(a): c,s=np.cos(a),np.sin(a); return np.array([[1,0,0],[0,c,-s],[0,s,c]])
    def Ry(a): c,s=np.cos(a),np.sin(a); return np.array([[c,0,s],[0,1,0],[-s,0,c]])
    def Rz(a): c,s=np.cos(a),np.sin(a); return np.array([[c,-s,0],[s,c,0],[0,0,1]])

    root_pos = np.array([row['root_translateX'],
                         row['root_translateY'],
                         row['root_translateZ']]) * 10.0
    R_root = (Rz(np.radians(row['root_rotateZ']))
              @ Ry(np.radians(row['root_rotateY']))
              @ Rx(np.radians(row['root_rotateX'])))

    wp, wr = {}, {}
    for name, parent, offset, rots in G1_CHAIN:
        R_loc = np.eye(3)
        for axis, col in rots:
            a = np.radians(float(row.get(col, 0.0)))
            if axis == 'X': R_loc = R_loc @ Rx(a)
            elif axis == 'Y': R_loc = R_loc @ Ry(a)
            else:             R_loc = R_loc @ Rz(a)
        if parent is None:
            wp[name] = root_pos.copy()
            wr[name] = R_root.copy()
        else:
            off = np.array(offset, dtype=float)
            wp[name] = wp[parent] + wr[parent] @ off
            wr[name] = wr[parent] @ R_loc

    # ── Natural elbow bend override ───────────────────────────────────────
    def _rod(axis, angle):
        c, s = np.cos(angle), np.sin(angle)
        K = np.array([[0.,-axis[2],axis[1]],[axis[2],0.,-axis[0]],[-axis[1],axis[0],0.]])
        return np.eye(3) + s * K + (1 - c) * (K @ K)

    for side in ('left', 'right'):
        sh, el = wp[f'{side}_shoulder'], wp[f'{side}_elbow']
        u = el - sh;  u_len = np.linalg.norm(u)
        if u_len < 1e-6: continue
        u_dir = u / u_len
        raw = np.cross(u_dir, np.array([0., 0., 1.]))
        r_len = np.linalg.norm(raw)
        if r_len < 0.1:
            raw = np.cross(u_dir, np.array([1., 0., 0.]));  r_len = np.linalg.norm(raw) + 1e-8
        el_ax    = raw / r_len
        el_angle = -np.radians(float(row.get(f'{side}_elbow_joint_dof', 0.0)))
        wp[f'{side}_wrist'] = el + _rod(el_ax, el_angle) @ u_dir * 250.0

    # ── Fixed-yaw derotation (same convention for every sequence) ─────────
    # Use the provided fixed yaw; fall back to per-frame yaw if not given.
    yaw = fixed_yaw_rad if fixed_yaw_rad is not None else np.radians(row['root_rotateZ'])
    c, s = np.cos(-yaw), np.sin(-yaw)
    R_yaw_inv = np.array([[c, -s, 0.], [s, c, 0.], [0., 0., 1.]])

    result = {}
    for k, v in wp.items():
        vd = R_yaw_inv @ v
        result[k] = np.array([vd[1], vd[2], vd[0]])   # [lateral, height, forward]
    return result


def g1_fk_batch(viz_joints, df, sampled):
    """
    Run g1_fk for every sampled frame using a FIXED yaw from frame 0.
    This matches the effective behaviour of the jump motion (tiny yaw span)
    and correctly shows body rotation for motions like cartwheel.
    """
    fixed_yaw = np.radians(df.iloc[sampled[0]]['root_rotateZ'])
    out = np.zeros((len(sampled), len(viz_joints), 3))
    for fi, frame_idx in enumerate(sampled):
        wp = g1_fk(df.iloc[frame_idx], fixed_yaw_rad=fixed_yaw)
        for ji, jname in enumerate(viz_joints):
            out[fi, ji] = wp.get(jname, np.zeros(3))
    return out


# ── Load and compute ──────────────────────────────────────────────────────
print('Loading G1 CSV…')
df_g1 = pd.read_csv(g1_csv)
print(f'  {len(df_g1)} frames, {len(df_g1.columns)} columns')

print('Running G1 FK…')
pos_g1 = g1_fk_batch(G1_VIZ, df_g1, sampled)

wp0 = g1_fk(df_g1.iloc[0], fixed_yaw_rad=np.radians(df_g1.iloc[0]['root_rotateZ']))
print(f'  Frame-0 sanity: pelvis Y={wp0["pelvis"][1]:.0f} mm  '
      f'left_foot Y={wp0["left_foot"][1]:.0f} mm  '
      f'neck Y={wp0["neck"][1]:.0f} mm')

foot_g1_idx = [G1_VIZ.index('left_foot'), G1_VIZ.index('right_foot')]
pos_g1[:, :, 1] -= pos_g1[:, foot_g1_idx, 1].min()

print(f'  Y-range after floor-align: [{pos_g1[:,:,1].min():.0f}, {pos_g1[:,:,1].max():.0f}] mm')
print('G1 FK complete ✓')

Loading G1 CSV…
  1463 frames, 36 columns
Running G1 FK…
  Frame-0 sanity: pelvis Y=784 mm  left_foot Y=-24 mm  neck Y=1173 mm
  Y-range after floor-align: [0, 1376] mm
G1 FK complete ✓


In [81]:
# ── Per-panel axis limits ─────────────────────────────────────────────────
def axis_limits(arrays, pct=95):
    """
    Return (xlim, ylim, zlim) using percentile-based limits.
    pct=95 clips the top 5 % of positions so that inverted-body outliers
    (e.g. cartwheel peak) do not blow out the scale for standing frames.
    """
    import numpy as np
    all_x = np.concatenate([a[:,:,0].ravel() for a in arrays])
    all_y = np.concatenate([a[:,:,1].ravel() for a in arrays])
    all_z = np.concatenate([a[:,:,2].ravel() for a in arrays])
    xmin, xmax = np.percentile(all_x, 100-pct), np.percentile(all_x, pct)
    ymax        = np.percentile(all_y, pct)      # floor already at 0
    zmin, zmax = np.percentile(all_z, 100-pct), np.percentile(all_z, pct)
    cx = (xmin+xmax)/2; cz = (zmin+zmax)/2
    span = max(xmax-xmin, ymax, zmax-zmin) * 0.6
    return (cx-span, cx+span), (0, ymax*1.05), (cz-span, cz+span)

lim_g1   = axis_limits([pos_g1])            # G1 alone
lim_soma = axis_limits([pos_u, pos_p])      # SOMA Uniform + Proportional together


# ── Generic skeleton drawing ──────────────────────────────────────────────
def draw_skeleton(ax, pos_frame, bones, viz, color, alpha=1.0, lw=2.5):
    """pos_frame: (N_joints, 3) mm Y-up.  Plot: X→ax.X, Z→ax.Y(depth), Y→ax.Z(height)."""
    for b0, b1 in bones:
        if b0 in viz and b1 in viz:
            i0, i1 = viz.index(b0), viz.index(b1)
            p0, p1 = pos_frame[i0], pos_frame[i1]
            ax.plot([p0[0], p1[0]], [p0[2], p1[2]], [p0[1], p1[1]],
                    color=color, lw=lw, alpha=alpha)
    for p in pos_frame:
        ax.scatter([p[0]], [p[2]], [p[1]], color=color, s=15, alpha=alpha, depthshade=False)


def style_ax(ax, title, xlim, ylim, zlim):
    ax.set_xlim(xlim); ax.set_ylim(zlim); ax.set_zlim(ylim)
    ax.set_xlabel('X', color='w', fontsize=6, labelpad=1)
    ax.set_ylabel('Z', color='w', fontsize=6, labelpad=1)
    ax.set_zlabel('Height', color='w', fontsize=6, labelpad=1)
    ax.tick_params(colors='grey', labelsize=5)
    for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:
        pane.fill = False
    ax.grid(True, alpha=0.1)
    ax.set_title(title, color='w', fontsize=10, fontweight='bold', pad=5)
    ax.view_init(elev=12, azim=45)
    ax.set_facecolor('#0d1117')


# ── Colors ────────────────────────────────────────────────────────────────
C_G1   = '#66bb6a'   # green — G1 Robot
C_UNIF = '#4fc3f7'   # blue  — SOMA Uniform
C_PROP = '#ef9a9a'   # red   — SOMA Proportional


# ── Frame renderer ────────────────────────────────────────────────────────
def render_frame(fi):
    fig = plt.figure(figsize=(20, 5), facecolor='#0d1117')
    ax1 = fig.add_subplot(1, 4, 1, projection='3d')
    ax2 = fig.add_subplot(1, 4, 2, projection='3d')
    ax3 = fig.add_subplot(1, 4, 3, projection='3d')
    ax4 = fig.add_subplot(1, 4, 4, projection='3d')

    # Panel 1: G1 Robot — its own scale
    draw_skeleton(ax1, pos_g1[fi], G1_BONES, G1_VIZ,    C_G1)
    style_ax(ax1, 'G1 Robot (29 DOF)', *lim_g1)

    # Panel 2: SOMA Uniform — SOMA scale
    draw_skeleton(ax2, pos_u[fi],  BONES, VIZ_JOINTS, C_UNIF)
    style_ax(ax2, 'SOMA Uniform', *lim_soma)

    # Panel 3: SOMA Proportional — SOMA scale
    draw_skeleton(ax3, pos_p[fi],  BONES, VIZ_JOINTS, C_PROP)
    style_ax(ax3, 'SOMA Proportional', *lim_soma)

    # Panel 4: Overlay — SOMA Uniform vs Proportional only (same scale)
    draw_skeleton(ax4, pos_u[fi],  BONES, VIZ_JOINTS, C_UNIF, alpha=0.8, lw=2.2)
    draw_skeleton(ax4, pos_p[fi],  BONES, VIZ_JOINTS, C_PROP, alpha=0.8, lw=2.2)
    style_ax(ax4, 'Uniform vs Proportional', *lim_soma)
    ax4.legend(handles=[
        Line2D([0],[0], color=C_UNIF, lw=2, label='SOMA Uniform'),
        Line2D([0],[0], color=C_PROP, lw=2, label='SOMA Proportional'),
    ], loc='upper left', fontsize=8, facecolor='#1a1a2e', labelcolor='w', framealpha=0.7)

    frame_num = sampled[fi]
    plt.suptitle(
        f'G1 Robot vs SOMA Uniform vs SOMA Proportional  ·  {MOTION_NAME}\n'
        f'FK  ·  frame {frame_num}  ·  t = {frame_num / 120:.2f} s',
        color='w', fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout(pad=0.3)
    return fig


# ── Generate animated GIF ─────────────────────────────────────────────────
N = len(sampled)
print(f'Generating {N} frames…')
gif_frames = []
t0 = time.time()
for fi in range(N):
    if fi % 60 == 0:
        print(f'  {fi}/{N}  ({time.time()-t0:.0f}s)')
    fig = render_frame(fi)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=85, bbox_inches='tight', facecolor='#0d1117')
    buf.seek(0)
    gif_frames.append(Image.open(buf).copy())
    plt.close(fig)

OUT = '/home/grease/gam/g1_soma_comparison_fk.gif'
gif_frames[0].save(OUT, save_all=True, append_images=gif_frames[1:],
                   duration=50, loop=0, optimize=False)
mb = os.path.getsize(OUT) / 1e6
print(f'\n✅  {OUT}')
print(f'   {mb:.1f} MB  ·  {N} frames  ·  {time.time()-t0:.0f}s to generate')
print(f'   Layout: G1 Robot | SOMA Uniform | SOMA Proportional | Uniform vs Proportional overlay')

Generating 366 frames…
  0/366  (0s)
  60/366  (32s)
  60/366  (32s)
  120/366  (61s)
  120/366  (61s)
  180/366  (89s)
  180/366  (89s)
  240/366  (119s)
  240/366  (119s)
  300/366  (147s)
  300/366  (147s)
  360/366  (174s)
  360/366  (174s)

✅  /home/grease/gam/g1_soma_comparison_fk.gif
   24.1 MB  ·  366 frames  ·  181s to generate
   Layout: G1 Robot | SOMA Uniform | SOMA Proportional | Uniform vs Proportional overlay

✅  /home/grease/gam/g1_soma_comparison_fk.gif
   24.1 MB  ·  366 frames  ·  181s to generate
   Layout: G1 Robot | SOMA Uniform | SOMA Proportional | Uniform vs Proportional overlay


In [82]:
# ════════════════════════════════════════════════════════════════════════════
# EXAMPLE 2 — cartwheel_R_001  (contrasts with Example 1 jump)
# ════════════════════════════════════════════════════════════════════════════
MOTION2 = 'cartwheel_R_001__A135'

# ── Locate files ─────────────────────────────────────────────────────────
all_g1  = sorted(glob.glob(os.path.join(G1_DIR,               '**/*.csv'), recursive=True))
all_u   = sorted(glob.glob(os.path.join(SOMA_UNIFORM_DIR,      '**/*.bvh'), recursive=True))
all_p   = sorted(glob.glob(os.path.join(SOMA_PROPORTIONAL_DIR, '**/*.bvh'), recursive=True))

g1_csv2  = next(f for f in all_g1 if MOTION2 in f)
bvh_u2   = next(f for f in all_u  if MOTION2 in f)
bvh_p2   = next(f for f in all_p  if MOTION2 in f)
print(f'Motion       : {MOTION2}')
print(f'G1           : {g1_csv2}')
print(f'Uniform      : {bvh_u2}')
print(f'Proportional : {bvh_p2}')

# ── SOMA FK ───────────────────────────────────────────────────────────────
print('\nParsing SOMA BVH files…')
j_u2, o_u2, c_u2, p_u2, frames_u2 = parse_bvh(bvh_u2)
j_p2, o_p2, c_p2, p_p2, frames_p2 = parse_bvh(bvh_p2)
sampled2 = list(range(0, min(len(frames_u2), len(frames_p2)), FRAME_STEP))
print(f'  {len(frames_u2)} BVH frames  →  {len(sampled2)} sampled')

pos_u2 = fk_batch(VIZ_JOINTS, j_u2, o_u2, c_u2, p_u2, frames_u2, sampled2)
pos_p2 = fk_batch(VIZ_JOINTS, j_p2, o_p2, c_p2, p_p2, frames_p2, sampled2)
foot_idx = [VIZ_JOINTS.index(n) for n in ('LeftFoot', 'RightFoot')]
pos_u2[:, :, 1] -= pos_u2[:, foot_idx, 1].min()
pos_p2[:, :, 1] -= pos_p2[:, foot_idx, 1].min()

# ── G1 FK ─────────────────────────────────────────────────────────────────
print('Running G1 FK…')
df_g1_2 = pd.read_csv(g1_csv2)
pos_g1_2 = g1_fk_batch(G1_VIZ, df_g1_2, sampled2)
foot_g1_idx = [G1_VIZ.index('left_foot'), G1_VIZ.index('right_foot')]
pos_g1_2[:, :, 1] -= pos_g1_2[:, foot_g1_idx, 1].min()
print(f'  G1 Y-range: [{pos_g1_2[:,:,1].min():.0f}, {pos_g1_2[:,:,1].max():.0f}] mm')

# ── Axis limits ───────────────────────────────────────────────────────────
lim_g1_2   = axis_limits([pos_g1_2])
lim_soma_2 = axis_limits([pos_u2, pos_p2])

# ── Render + GIF ─────────────────────────────────────────────────────────
def render_frame2(fi):
    fig = plt.figure(figsize=(20, 5), facecolor='#0d1117')
    ax1 = fig.add_subplot(1, 4, 1, projection='3d')
    ax2 = fig.add_subplot(1, 4, 2, projection='3d')
    ax3 = fig.add_subplot(1, 4, 3, projection='3d')
    ax4 = fig.add_subplot(1, 4, 4, projection='3d')

    draw_skeleton(ax1, pos_g1_2[fi], G1_BONES, G1_VIZ,    C_G1)
    draw_skeleton(ax2, pos_u2[fi],   BONES,    VIZ_JOINTS, C_UNIF)
    draw_skeleton(ax3, pos_p2[fi],   BONES,    VIZ_JOINTS, C_PROP)
    draw_skeleton(ax4, pos_u2[fi],   BONES,    VIZ_JOINTS, C_UNIF, alpha=0.8, lw=2.2)
    draw_skeleton(ax4, pos_p2[fi],   BONES,    VIZ_JOINTS, C_PROP, alpha=0.8, lw=2.2)

    style_ax(ax1, 'G1 Robot (29 DOF)',       *lim_g1_2)
    style_ax(ax2, 'SOMA Uniform',             *lim_soma_2)
    style_ax(ax3, 'SOMA Proportional',        *lim_soma_2)
    style_ax(ax4, 'Uniform vs Proportional',  *lim_soma_2)
    ax4.legend(handles=[
        Line2D([0],[0], color=C_UNIF, lw=2, label='SOMA Uniform'),
        Line2D([0],[0], color=C_PROP, lw=2, label='SOMA Proportional'),
    ], loc='upper left', fontsize=8, facecolor='#1a1a2e', labelcolor='w', framealpha=0.7)

    frame_num = sampled2[fi]
    plt.suptitle(
        f'G1 Robot vs SOMA Uniform vs SOMA Proportional  ·  {MOTION2}\n'
        f'FK  ·  frame {frame_num}  ·  t = {frame_num / 120:.2f} s',
        color='w', fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout(pad=0.3)
    return fig

N2 = len(sampled2)
print(f'\nGenerating {N2} frames…')
gif2 = []
t0 = time.time()
for fi in range(N2):
    if fi % 50 == 0:
        print(f'  {fi}/{N2}  ({time.time()-t0:.0f}s)')
    fig = render_frame2(fi)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=85, bbox_inches='tight', facecolor='#0d1117')
    buf.seek(0)
    gif2.append(Image.open(buf).copy())
    plt.close(fig)

OUT2 = '/home/grease/gam/g1_soma_cartwheel_fk.gif'
gif2[0].save(OUT2, save_all=True, append_images=gif2[1:], duration=50, loop=0, optimize=False)
mb2 = os.path.getsize(OUT2) / 1e6
print(f'\n✅  {OUT2}')
print(f'   {mb2:.1f} MB  ·  {N2} frames  ·  {time.time()-t0:.0f}s to generate')

Motion       : cartwheel_R_001__A135
G1           : /home/grease/ego_dataset/work_bearlu/data/bones-studio-seed/g1/csv/230123/cartwheel_R_001__A135.csv
Uniform      : /home/grease/ego_dataset/work_bearlu/data/bones-studio-seed/soma_uniform/bvh/230123/cartwheel_R_001__A135.bvh
Proportional : /home/grease/ego_dataset/work_bearlu/data/bones-studio-seed/soma_proportional/bvh/230123/cartwheel_R_001__A135.bvh

Parsing SOMA BVH files…
  851 BVH frames  →  213 sampled
Running G1 FK…
  G1 Y-range: [0, 1754] mm

Generating 213 frames…
  0/213  (0s)
  50/213  (22s)
  100/213  (49s)
  150/213  (73s)
  200/213  (96s)

✅  /home/grease/gam/g1_soma_cartwheel_fk.gif
   12.9 MB  ·  213 frames  ·  103s to generate


In [83]:
# ════════════════════════════════════════════════════════════════════════════
# EXAMPLE 3 — sit_on_heels_loop_009  (kneeling / sitting motion)
# Contrasts with:  Ex1 = vertical jump,  Ex2 = full-body cartwheel
# Motion: person sits down onto heels and holds the kneeling pose in a loop
# ════════════════════════════════════════════════════════════════════════════
MOTION3 = 'sit_on_heels_loop_009__A548'

# ── Locate files ──────────────────────────────────────────────────────────
g1_csv3  = next(f for f in all_g1 if MOTION3 in f)
bvh_u3   = next(f for f in all_u  if MOTION3 in f)
bvh_p3   = next(f for f in all_p  if MOTION3 in f)
print(f'Motion       : {MOTION3}')
print(f'G1           : {g1_csv3}')
print(f'Uniform      : {bvh_u3}')
print(f'Proportional : {bvh_p3}')

# ── SOMA FK ───────────────────────────────────────────────────────────────
print('\nParsing SOMA BVH files…')
j_u3, o_u3, c_u3, p_u3, frames_u3 = parse_bvh(bvh_u3)
j_p3, o_p3, c_p3, p_p3, frames_p3 = parse_bvh(bvh_p3)
sampled3 = list(range(0, min(len(frames_u3), len(frames_p3)), FRAME_STEP))
print(f'  {len(frames_u3)} BVH frames  →  {len(sampled3)} sampled')

pos_u3 = fk_batch(VIZ_JOINTS, j_u3, o_u3, c_u3, p_u3, frames_u3, sampled3)
pos_p3 = fk_batch(VIZ_JOINTS, j_p3, o_p3, c_p3, p_p3, frames_p3, sampled3)
foot_idx = [VIZ_JOINTS.index(n) for n in ('LeftFoot', 'RightFoot')]
pos_u3[:, :, 1] -= pos_u3[:, foot_idx, 1].min()
pos_p3[:, :, 1] -= pos_p3[:, foot_idx, 1].min()
print(f'  SOMA height range: [{pos_u3[:,:,1].min():.0f}, {pos_u3[:,:,1].max():.0f}] mm')

# ── G1 FK ─────────────────────────────────────────────────────────────────
print('Running G1 FK…')
df_g1_3 = pd.read_csv(g1_csv3)
pos_g1_3 = g1_fk_batch(G1_VIZ, df_g1_3, sampled3)
foot_g1_idx3 = [G1_VIZ.index('left_foot'), G1_VIZ.index('right_foot')]
pos_g1_3[:, :, 1] -= pos_g1_3[:, foot_g1_idx3, 1].min()
print(f'  G1 height range:   [{pos_g1_3[:,:,1].min():.0f}, {pos_g1_3[:,:,1].max():.0f}] mm')

# ── Axis limits ───────────────────────────────────────────────────────────
lim_g1_3   = axis_limits([pos_g1_3])
lim_soma_3 = axis_limits([pos_u3, pos_p3])

# ── Render + GIF ──────────────────────────────────────────────────────────
def render_frame3(fi):
    fig = plt.figure(figsize=(20, 5), facecolor='#0d1117')
    ax1 = fig.add_subplot(1, 4, 1, projection='3d')
    ax2 = fig.add_subplot(1, 4, 2, projection='3d')
    ax3 = fig.add_subplot(1, 4, 3, projection='3d')
    ax4 = fig.add_subplot(1, 4, 4, projection='3d')

    draw_skeleton(ax1, pos_g1_3[fi], G1_BONES, G1_VIZ,    C_G1)
    draw_skeleton(ax2, pos_u3[fi],   BONES,    VIZ_JOINTS, C_UNIF)
    draw_skeleton(ax3, pos_p3[fi],   BONES,    VIZ_JOINTS, C_PROP)
    draw_skeleton(ax4, pos_u3[fi],   BONES,    VIZ_JOINTS, C_UNIF, alpha=0.8, lw=2.2)
    draw_skeleton(ax4, pos_p3[fi],   BONES,    VIZ_JOINTS, C_PROP, alpha=0.8, lw=2.2)

    style_ax(ax1, 'G1 Robot (29 DOF)',      *lim_g1_3)
    style_ax(ax2, 'SOMA Uniform',            *lim_soma_3)
    style_ax(ax3, 'SOMA Proportional',       *lim_soma_3)
    style_ax(ax4, 'Uniform vs Proportional', *lim_soma_3)
    ax4.legend(handles=[
        Line2D([0],[0], color=C_UNIF, lw=2, label='SOMA Uniform'),
        Line2D([0],[0], color=C_PROP, lw=2, label='SOMA Proportional'),
    ], loc='upper left', fontsize=8, facecolor='#1a1a2e', labelcolor='w', framealpha=0.7)

    frame_num = sampled3[fi]
    plt.suptitle(
        f'G1 Robot vs SOMA  ·  {MOTION3}  (kneeling / sit-on-heels loop)\n'
        f'FK  ·  frame {frame_num}  ·  t = {frame_num / 120:.2f} s',
        color='w', fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout(pad=0.3)
    return fig

N3 = len(sampled3)
print(f'\nGenerating {N3} frames…')
gif3 = []
t0 = time.time()
for fi in range(N3):
    if fi % 40 == 0:
        print(f'  {fi}/{N3}  ({time.time()-t0:.0f}s)')
    fig = render_frame3(fi)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=85, bbox_inches='tight', facecolor='#0d1117')
    buf.seek(0)
    gif3.append(Image.open(buf).copy())
    plt.close(fig)

OUT3 = '/home/grease/gam/g1_soma_sit_on_heels_fk.gif'
gif3[0].save(OUT3, save_all=True, append_images=gif3[1:], duration=50, loop=0, optimize=False)
mb3 = os.path.getsize(OUT3) / 1e6
print(f'\n✅  {OUT3}')
print(f'   {mb3:.1f} MB  ·  {N3} frames  ·  {time.time()-t0:.0f}s')

Motion       : sit_on_heels_loop_009__A548
G1           : /home/grease/ego_dataset/work_bearlu/data/bones-studio-seed/g1/csv/240918/sit_on_heels_loop_009__A548.csv
Uniform      : /home/grease/ego_dataset/work_bearlu/data/bones-studio-seed/soma_uniform/bvh/240918/sit_on_heels_loop_009__A548.bvh
Proportional : /home/grease/ego_dataset/work_bearlu/data/bones-studio-seed/soma_proportional/bvh/240918/sit_on_heels_loop_009__A548.bvh

Parsing SOMA BVH files…
  619 BVH frames  →  155 sampled
  SOMA height range: [-3, 82] mm
Running G1 FK…
  G1 height range:   [-222, 465] mm

Generating 155 frames…
  0/155  (0s)
  40/155  (22s)
  80/155  (42s)
  120/155  (61s)

✅  /home/grease/gam/g1_soma_sit_on_heels_fk.gif
   11.5 MB  ·  155 frames  ·  78s


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# EXAMPLE 4 — g_m  3-D skeleton (sparse VR trackers)
#
# g_m layout  (11 columns):
#   [0:3]  Head  XYZ   ●  (SMPL joint 15)
#   [3:6]  L Wrist XYZ ●  (SMPL joint 20)
#   [6:9]  R Wrist XYZ ●  (SMPL joint 21)
#   [9]    mean lower-body G1 angle  (scalar)
#   [10]   last lower-body G1 joint  (scalar)
#
# These are the 3 positions a Pico headset + 2 controllers would stream
# at deployment time (no Pico Swift body trackers needed).
# ════════════════════════════════════════════════════════════════════════════
PROCESSED = '/home/grease/ego_dataset/work_bearlu/data/bones-studio-processed'

MOTIONS_GM = {
    'bottle_throw\n(throw)':     'drinking_bottle_throw_270_R_001__A548.npz',
    'crossed_arms\n(idle)':      'crossed_arms_idle_R_002__A549.npz',
    'crouch_walk\n(locomotion)': 'h_b_w_crouch_270_loop_002__A548.npz',
}

TRACKER_COLORS = {'Head': '#ffffff', 'L Wrist': '#4fc3f7', 'R Wrist': '#ef9a9a'}
TRACKER_SLICES = {'Head': slice(0,3), 'L Wrist': slice(3,6), 'R Wrist': slice(6,9)}
TRACKER_BONES  = [('Head','L Wrist'), ('Head','R Wrist'), ('L Wrist','R Wrist')]

# ── Load + sample ─────────────────────────────────────────────────────────
gm_seqs, gm_idx = {}, {}
for label, fname in MOTIONS_GM.items():
    d = np.load(os.path.join(PROCESSED, fname))
    gm = d['g_m']
    idx = list(range(0, len(gm), 4))
    gm_seqs[label] = gm;  gm_idx[label] = idx
    print(f"  {label.replace(chr(10),' '):<25s}  {len(gm)} frames → {len(idx)} sampled")

N_GM = min(len(v) for v in gm_idx.values())
print(f"  Generating {N_GM} frames…")

# ── Per-motion axis limits ─────────────────────────────────────────────────
def gm_limits(gm, idx):
    pts = gm[idx, :9].reshape(-1, 3)
    cx = (pts[:,0].max()+pts[:,0].min())/2
    cy = (pts[:,1].max()+pts[:,1].min())/2
    cz = (pts[:,2].max()+pts[:,2].min())/2
    sp = max(pts[:,0].max()-pts[:,0].min(),
             pts[:,1].max()-pts[:,1].min(),
             pts[:,2].max()-pts[:,2].min()) * 0.65
    return (cx-sp,cx+sp), (cy-sp,cy+sp), (cz-sp,cz+sp)

gm_lims = {l: gm_limits(gm_seqs[l], gm_idx[l]) for l in MOTIONS_GM}

def style_gm_ax(ax, title, xlim, ylim, zlim):
    ax.set_xlim(xlim); ax.set_ylim(zlim); ax.set_zlim(ylim)
    ax.set_xlabel('X', color='w', fontsize=6, labelpad=1)
    ax.set_ylabel('Z', color='w', fontsize=6, labelpad=1)
    ax.set_zlabel('Y', color='w', fontsize=6, labelpad=1)
    ax.tick_params(colors='grey', labelsize=5)
    for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]: pane.fill = False
    ax.grid(True, alpha=0.1)
    ax.set_title(title, color='w', fontsize=9, fontweight='bold', pad=5)
    ax.view_init(elev=12, azim=45)
    ax.set_facecolor('#0d1117')

def draw_gm(ax, gm_row):
    pts = {n: gm_row[sl] for n, sl in TRACKER_SLICES.items()}
    for b0, b1 in TRACKER_BONES:
        p0, p1 = pts[b0], pts[b1]
        ax.plot([p0[0],p1[0]], [p0[2],p1[2]], [p0[1],p1[1]], color='#888', lw=1.5, alpha=0.7)
    for name, p in pts.items():
        ax.scatter([p[0]], [p[2]], [p[1]],
                   color=TRACKER_COLORS[name], s=120 if name=='Head' else 70,
                   depthshade=False, zorder=5)

def render_gm_frame(fi):
    fig = plt.figure(figsize=(20, 5), facecolor='#0d1117')
    labels = list(MOTIONS_GM.keys())
    for col, label in enumerate(labels):
        gm   = gm_seqs[label]
        fidx = gm_idx[label][fi]
        xl, yl, zl = gm_lims[label]
        ax = fig.add_subplot(1, 4, col+1, projection='3d')
        draw_gm(ax, gm[fidx])
        ax.text2D(0.05, 0.02, f'lower: {gm[fidx,9]:.1f}°',
                  transform=ax.transAxes, color='#ffd54f', fontsize=7)
        style_gm_ax(ax, label, xl, yl, zl)

    # Overlay
    ax4 = fig.add_subplot(1, 4, 4, projection='3d')
    ov_colors = ['#4fc3f7', '#66bb6a', '#ef9a9a']
    all_pts = []
    for label, oc in zip(labels, ov_colors):
        gm = gm_seqs[label];  fidx = gm_idx[label][fi]
        pts = {n: gm[fidx, sl] for n, sl in TRACKER_SLICES.items()}
        for b0, b1 in TRACKER_BONES:
            p0, p1 = pts[b0], pts[b1]
            ax4.plot([p0[0],p1[0]], [p0[2],p1[2]], [p0[1],p1[1]], color=oc, lw=1.5, alpha=0.7)
        for p in pts.values():
            ax4.scatter([p[0]], [p[2]], [p[1]], color=oc, s=50, depthshade=False)
            all_pts.append(p)
    all_pts = np.array(all_pts)
    cx, cy, cz = all_pts[:,0].mean(), all_pts[:,1].mean(), all_pts[:,2].mean()
    sp = max(all_pts[:,0].max()-all_pts[:,0].min(),
             all_pts[:,1].max()-all_pts[:,1].min(),
             all_pts[:,2].max()-all_pts[:,2].min()) * 0.65 + 1e-3
    for pane in [ax4.xaxis.pane, ax4.yaxis.pane, ax4.zaxis.pane]: pane.fill = False
    ax4.set_xlim(cx-sp,cx+sp); ax4.set_ylim(cz-sp,cz+sp); ax4.set_zlim(cy-sp,cy+sp)
    ax4.tick_params(colors='grey', labelsize=5); ax4.grid(True, alpha=0.1)
    ax4.set_facecolor('#0d1117'); ax4.view_init(elev=12, azim=45)
    ax4.set_title('Overlay', color='w', fontsize=9, fontweight='bold', pad=5)
    ax4.legend(handles=[Line2D([0],[0], color=c, lw=2, label=l.replace('\n',' '))
                        for l, c in zip(labels, ov_colors)],
               loc='upper left', fontsize=7, facecolor='#1a1a2e', labelcolor='w', framealpha=0.7)

    fidx0 = gm_idx[labels[0]][fi]
    plt.suptitle(
        f'g_m — Sparse VR Trackers (● Head  ● L Wrist  ● R Wrist)  ·  frame {fidx0}  ·  t = {fidx0/120:.2f}s\n'
        f'Input as seen by the model at inference time  ·  bones connect trackers  ·  lower-body is scalar only',
        color='w', fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout(pad=0.3)
    return fig

# ── Generate GIF ───────────────────────────────────────────────────────────
gm_frames = []
t0 = time.time()
for fi in range(N_GM):
    if fi % 40 == 0: print(f'  {fi}/{N_GM}  ({time.time()-t0:.0f}s)')
    fig = render_gm_frame(fi)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=90, bbox_inches='tight', facecolor='#0d1117')
    buf.seek(0)
    gm_frames.append(Image.open(buf).copy())
    plt.close(fig)

OUT_GM = '/home/grease/gam/gm_skeleton_3d.gif'
gm_frames[0].save(OUT_GM, save_all=True, append_images=gm_frames[1:],
                  duration=50, loop=0, optimize=False)
mb_gm = os.path.getsize(OUT_GM) / 1e6
print(f'\n✅  {OUT_GM}')
print(f'   {mb_gm:.1f} MB  ·  {N_GM} frames  ·  {time.time()-t0:.0f}s')